In [0]:
# %python
# =========================================
# SILVER — Dimensão Municípios (IBGE)
# Fonte: Bronze ibge_municipios.parquet (mais recente)
# Saída: silver.dim_ibge_municipios (Delta)
# Chave única: municipio_id
# =========================================
import io, sys
from pathlib import Path

repo = Path.cwd()
while not repo.name.startswith("postech-aisc") and repo.parent != repo:
    repo = repo.parent
sys.path.insert(0, str(repo))

from src.config.settings import AZURE_STORAGE_ACCOUNT, AZURE_STORAGE_KEY, BRONZE_CONTAINER
from azure.storage.blob import BlobServiceClient
import pandas as pd

# ---------- 1. LER DO BRONZE (leitura dinâmica do arquivo mais recente) ----------
conn_str = (f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};"
            f"AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net")
blob_service_client = BlobServiceClient.from_connection_string(conn_str)
container_client = blob_service_client.get_container_client(BRONZE_CONTAINER)

prefixo = "ibge_municipios.parquet"
candidatos = [b.name for b in container_client.list_blobs() if b.name.endswith(prefixo)]

if not candidatos:
    raise ValueError(f"Nenhum arquivo '{prefixo}' encontrado no container '{BRONZE_CONTAINER}'.")

arquivo_mais_recente = sorted(candidatos)[-1]
print(f"[INFO] Arquivo selecionado: {arquivo_mais_recente}")

data = container_client.get_blob_client(arquivo_mais_recente).download_blob().readall()
pdf = pd.read_parquet(io.BytesIO(data))

# ---------- 2. TRANSFORMAÇÕES ----------
# Padronizar municipio_id (código IBGE como string, sem espaços)
pdf["municipio_id"] = pdf["municipio_id"].astype(str).str.strip()
# Padronizar estado_sigla (chave de relacionamento com estados)
pdf["estado_sigla"] = pdf["estado_sigla"].astype(str).str.strip().str.upper()
# Metadados de rastreabilidade (princípio da arquitetura)
pdf["ingested_at"] = pdf["_ingested_at"]
pdf["source"] = f"bronze/{arquivo_mais_recente}"
pdf["version"] = "1.0"

# ---------- 3. DATA QUALITY ----------
chaves = ["municipio_id"]
nulos_chave = pdf[chaves].isna().sum().sum()
dups = pdf.duplicated(subset=chaves).sum()
print(f"[DQ] Nulos na chave: {nulos_chave}")
print(f"[DQ] Duplicados na chave única: {dups}")
print(f"[DQ] Municípios encontrados: {len(pdf)}")

# ---------- 3.5 PERSISTIR DQ NO MONITORAMENTO ----------
spark.sql("CREATE DATABASE IF NOT EXISTS monitoring")
spark.sql("""
  CREATE TABLE IF NOT EXISTS monitoring.dq_results (
    table_name STRING, rule STRING, status STRING,
    records_checked BIGINT, failures BIGINT, run_at TIMESTAMP
  ) USING DELTA
""")

from pyspark.sql import functions as F

registros = [
    ("silver.dim_ibge_municipios", "completude_chave",
     "PASS" if nulos_chave == 0 else "FAIL", int(len(pdf)), int(nulos_chave)),
    ("silver.dim_ibge_municipios", "unicidade_chave_unica",
     "PASS" if dups == 0 else "FAIL", int(len(pdf)), int(dups)),
]
df_dq = spark.createDataFrame(registros,
    ["table_name", "rule", "status", "records_checked", "failures"]) \
    .withColumn("run_at", F.current_timestamp())
df_dq.write.mode("append").saveAsTable("monitoring.dq_results")

# ---------- 4. GRAVAR EM DELTA (overwriteSchema evita conflito de schema) ----------
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
df = spark.createDataFrame(pdf)
df = df.drop("_ingested_at", "_source_table")
df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver.dim_ibge_municipios")

print(f"\n[OK] Silver dim_ibge_municipios gravada | Registros: {df.count()}")

In [0]:
%sql
SELECT DISTINCT source, ingested_at
FROM silver.dim_ibge_municipios;






